# Galea analysis

This notebook loads a session **with derived signals**, opens the signal browser, and adds analysis GUIs that run on the **browser's time range**.

* **Derived** channels (browser group "Derived"):
  * `EMG<n> env`: EMG linear envelope (muscle activity).
  * `Linear accel |a|`, `Gyro |ω|`, `Pitch`, `Roll`: head motion from IMU fusion.
  * `Heart rate`, `Breathing (PPG)`, `Breathing phase`: computed from the PPG.
* Select a range in the browser (zoom, or *start / window* + **go**), then press **clip to window** to fix it as the analysis range. The setup period is skipped by default (see below).
* **Pair analysis:** pick channels A and B, a preprocessing and an analysis (cross-correlation or coherence), each with its own options. **run** recomputes on the current range; **export figure** saves a PNG to `exports/`.

In [ ]:
%load_ext autoreload
%autoreload 2
from galea import load_session, resolve_data, settle_time
from galea.browser import SignalBrowser
from galea.pair_gui import PairGUI

DATA = None          # session folder; None → $GALEA_DATA → first session under ./data

## Load

In [ ]:
s = load_session(resolve_data(DATA), derived=True)
t_settle = settle_time(s)
print(f"{s.name}: start {s.t0_wall():%Y-%m-%d %H:%M:%S %z}, duration {s.exg.t.iloc[-1]:.1f} s, "
      f"usable from t ≈ {t_settle:.0f} s")
for r in s.resp_info:
    print(f"  breathing run {r['t0']:.1f}–{r['t1']:.1f} s: {r['beats']} beats, "
          f"{r['breaths_per_min']:.1f} breaths/min, HR–pulse-amplitude r = {r['hr_amp_r']:.2f}")

## Browser
It starts clipped to the usable range. Press **unclip** to see the whole recording.

In [ ]:
b = SignalBrowser(s, channels=["EEG7", "EOG0", "PPG IR", "Heart rate", "Breathing (PPG)", "Linear accel |a|"])
b.set_clip(t_settle, float(s.exg.t.iloc[-1]))
b

## Pair analysis

In [ ]:
pg = PairGUI(b, "Heart rate", "Breathing (PPG)", analysis="coherence")
pg